In [1]:
# --- BLOQUE 0: INFRAESTRUCTURA ---
import sys
import subprocess
import warnings

warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package])

try:
    from astroquery.heasarc import Heasarc
    import astropy
    import scipy
    import matplotlib
except ImportError:
    print("[SYSTEM] Desplegando Astroquery...")
    install("astroquery")
    install("astropy")
    install("scipy")
    install("matplotlib")
    from astroquery.heasarc import Heasarc

import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit

print("[SYSTEM] Iniciando Protocolo ASTRO-MINER ULTIMATE (Select All)...")

# --- BLOQUE 1: EXTRACCIÓN MASIVA ---
heasarc = Heasarc()
TARGET_TABLE = 'fermilgrb'

print(f"[QUERY] Descargando TODAS las columnas de '{TARGET_TABLE}'...")

try:
    # TÁCTICA FINAL:
    # 1. object_name='GRB*': Busca todo lo que empiece por GRB (casi todo el catálogo).
    # 2. fields='All': OBLIGA a HEASARC a mandar todas las columnas (incluida REDSHIFT).
    # 3. result_max=3000: Asegura que no corte la lista.
    table = heasarc.query_object(object_name='GRB*', mission=TARGET_TABLE,
                                 fields='All', result_max=3000)

    df = table.to_pandas()
    # Normalizar nombres de columnas a mayúsculas
    df.columns = [c.upper() for c in df.columns]
    print(f"[SUCCESS] Tabla recuperada. Dimensiones: {df.shape}")

except Exception as e:
    print(f"[CRITICAL] Error fatal de conexión: {e}")
    sys.exit()

# --- BLOQUE 2: SELECCIÓN DE VARIABLES ---
# Usamos las columnas que aparecieron en tu propio volcado de logs.
print("[MAP] Seleccionando variables físicas...")

# Columnas candidatas (basado en tu lista)
col_z = 'REDSHIFT'
col_t = 'TL0' # Tiempo de inicio de emisión LAT (>100 MeV). Proxy de retraso.
col_name = 'NAME'

# Verificación
if col_z not in df.columns or col_t not in df.columns:
    print(f"[ERROR] Faltan columnas clave. Disponibles: {list(df.columns)}")
    sys.exit()

# Extracción
raw_z = df[col_z]
raw_t = df[col_t]
raw_n = df[col_name]

# Filtros Físicos
# 1. Redshift conocido (> 0)
# 2. TL0 definido (indica que el LAT detectó algo)
mask = (raw_z > 0) & (np.isfinite(raw_z)) & (np.isfinite(raw_t))

clean_z = raw_z[mask].values
clean_t = raw_t[mask].values
clean_names = raw_n[mask].values

print(f"[DATA] Eventos con Redshift y Tiempo LAT: {len(clean_z)}")

if len(clean_z) < 3:
    print("[WARN] Pocos datos. ¿Está la columna REDSHIFT vacía en esta tabla?")

# --- BLOQUE 3: CÁLCULO Y VISUALIZACIÓN ---
print("[PHYS] Calculando Time-of-Flight...")

# K-Factor (Distancia)
def get_k(z):
    z_grid = np.linspace(0, z, 50)
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

if len(clean_z) > 0:
    K_factors = np.array([get_k(z) for z in clean_z])
    lags = clean_t # Usamos el tiempo de inicio como retraso observable
else:
    K_factors = []
    lags = []

# Regresión Lineal (Test de LIV)
def model(k, m, c):
    return m * k + c

m_fit, m_err = 0, 0
if len(K_factors) > 2:
    try:
        # Ajuste robusto (Sigma Clipping para quitar outliers astrofísicos)
        q1, q3 = np.percentile(lags, [25, 75])
        iqr = q3 - q1
        robust_mask = (lags > q1 - 2*iqr) & (lags < q3 + 2*iqr)

        if np.sum(robust_mask) > 2:
            popt, pcov = curve_fit(model, K_factors[robust_mask], lags[robust_mask])
            m_fit = popt[0]
            m_err = np.sqrt(np.diag(pcov))[0]
            plot_k = K_factors[robust_mask]
            plot_lags = lags[robust_mask]
            plot_names = clean_names[robust_mask]
        else:
            plot_k, plot_lags, plot_names = K_factors, lags, clean_names

    except: pass
else:
    plot_k, plot_lags, plot_names = [], [], []

# Plot
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

if len(plot_k) > 0:
    plt.scatter(plot_k, plot_lags, c='cyan', s=80, alpha=0.8, label='Inicio Emisión LAT')

    if m_fit != 0:
        x_plot = np.linspace(0, max(plot_k)*1.1, 50)
        plt.plot(x_plot, model(x_plot, m_fit, popt[1]), color='magenta',
                 linestyle='--', lw=2, label=f'Tendencia LIV (m={m_fit:.2f})')

    plt.axhline(0, color='white', linestyle=':', label='Simultaneidad (Einstein)')

    # Etiquetas
    for i in range(len(plot_names)):
        if plot_k[i] > 2 or plot_lags[i] > 50:
            try: n_str = str(plot_names[i])
            except: n_str = "GRB"
            plt.annotate(n_str, (plot_k[i], plot_lags[i]), color='yellow', fontsize=8)

plt.xlabel(r'Factor de Distancia Cosmológica $K(z)$')
plt.ylabel(r'Tiempo de Inicio LAT $T_{L0}$ (s)')
plt.title('Test de Gravedad Cuántica: Retraso vs Distancia\nDatos: HEASARC fermilgrb', color='white')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n[RESULT] Pendiente observada: {m_fit:.4f} +/- {m_err:.4f} s")
if m_fit > 0 and m_fit > 2*m_err:
    print(">> POSIBLE SEÑAL: Los fotones lejanos llegan más tarde.")
else:
    print(">> RESULTADO NULO: No hay correlación obvia con la distancia.")

[SYSTEM] Desplegando Astroquery...
[SYSTEM] Iniciando Protocolo ASTRO-MINER ULTIMATE (Select All)...
[QUERY] Descargando TODAS las columnas de 'fermilgrb'...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[CRITICAL] Error fatal de conexión: Unable to find coordinates for name 'GRB*' using https://cds.unistra.fr/cgi-bin/nph-sesame/SNV?GRB%2A
Traceback (most recent call last):
  File "/tmp/ipython-input-3820465436.py", line 42, in <cell line: 0>
    table = heasarc.query_object(object_name='GRB*', mission=TARGET_TABLE,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/utils/decorators.py", line 143, in deprecated_func
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astroquery/heasarc/core.py", line 463, in query_object
    pos = coordinates.SkyCoord.from_name(object_name)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/coordinates/sky_coordinate.py", line 1905, in from_name
    icrs_coord = get_icrs_coordinates(name, parse, cache=cache)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

TypeError: object of type 'NoneType' has no len()